In [1]:
import os
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
GPU = 0
import os, gc, re, ast, io, contextlib, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
set_seed(42)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True,
    torch_dtype=torch.float16, device_map={'': GPU}, attn_implementation='sdpa')
model.eval()
assert not getattr(model, 'is_quantized', False)
assert {p.dtype for p in model.parameters() if p.is_floating_point()} == {torch.float16}
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})
@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k,v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None,
        max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
def release_model():
    global model
    del model
    gc.collect()
    torch.cuda.empty_cache()


In [6]:
if 'model' not in globals() and 'mathqa_accuracy' in globals():
    print(f'MathQA | FP16 | 准确率: {mathqa_accuracy:.2%}')
else:
    test = load_from_disk(ROOT / 'mathqa_fixed_1000')
    demos = load_from_disk(ROOT / 'mathqa_fixed_5shot')
    assert len(test) == 1000 and len(demos) == 5
    assert not ({x['Problem'].strip() for x in test} & {x['Problem'].strip() for x in demos})
    def question(row):
        return f"Question: {row['Problem'].strip()}\nOptions: {row['options'].strip()}"
    def messages(row):
        msg = [{'role':'system','content':'Solve the multiple-choice math problem. Explain your reasoning concisely, then end with exactly: Final answer: <letter> where <letter> is a, b, c, d, or e.'}]
        for d in demos:
            msg += [{'role':'user','content':question(d)}, {'role':'assistant','content':d['Rationale'].strip() + '\nFinal answer: ' + d['correct'].strip().lower()}]
        return msg + [{'role':'user','content':question(row)}]
    def parse_answer(text):
        matches = re.findall(r'(?im)^\s*(?:\*\*)?final answer\s*:\s*(?:\*\*)?\s*[\(\[]?([a-e])\b', text)
        if matches: return matches[-1].lower()
        matches = re.findall(r'(?i)\b(?:the\s+)?(?:correct\s+)?answer\s*(?:is|:)\s*(?:\*\*)?[\(\[]?([a-e])\b', text)
        if matches: return matches[-1].lower()
        return text.strip().lower() if re.fullmatch('[a-e]', text.strip().lower()) else None
    correct = 0
    processed = 0
    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start+4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)
        correct += sum(parse_answer(text) == row['correct'].strip().lower() for text,row in zip(generated,rows))
        processed += len(rows)
    accuracy = correct / len(test)
    mathqa_accuracy = accuracy
    print(f'MathQA | FP16 | {correct}/{len(test)} | 准确率: {accuracy:.2%}')
    release_model()
    
    

MathQA | FP16 | 准确率: 48.80%


In [ ]:
quant_code = r'''
import os, sysconfig
from pathlib import Path
_nvroot = Path(sysconfig.get_paths()['purelib']) / 'nvidia'
os.environ['CPATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/include')))
os.environ['LIBRARY_PATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/lib')))
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6;8.9'
GPU = 0
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import gc, re, ast, textwrap, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
from optimum.quanto import quantize, freeze, qint8, quantization_map
from optimum.quanto.nn import QModuleMixin
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})

def release_model():
    global model
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

def load_quantized_model(bits):
    global model
    assert bits == 8
    release_model()
    set_seed(42)
    model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, local_files_only=True, torch_dtype=torch.float16, device_map={'': GPU}, attn_implementation='sdpa').eval()
    expected = [name for name, module in model.named_modules() if name.startswith('model.layers.') and isinstance(module, torch.nn.Linear)]
    assert len(expected) == model.config.num_hidden_layers * 7
    qtype = qint8
    quantize(model, weights=qtype, activations=None, include=expected)
    freeze(model)
    config = quantization_map(model)
    assert set(config) == set(expected)
    assert all((c == {'weights': f'qint{bits}', 'activations': 'none'} for c in config.values()))
    modules = [model.get_submodule(name) for name in expected]
    assert all((m.frozen and m.weight.dtype == torch.float16 for m in modules))
    assert model.model.embed_tokens.weight.dtype == model.lm_head.weight.dtype == torch.float16
    seen = set()

    def check_input(module, args):
        assert args[0].dtype == torch.float16, 'Activation is not FP16'
        seen.add(id(module))
    hooks = [m.register_forward_pre_hook(check_input) for m in modules]
    try:
        sample = tokenizer('Hello', return_tensors='pt').to(model.device)
        with torch.inference_mode():
            logits = model(**sample, use_cache=False).logits
        assert len(seen) == len(modules) and torch.isfinite(logits).all()
    finally:
        for hook in hooks:
            hook.remove()
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None, max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def run_evaluation():
    test = load_from_disk(ROOT / 'mathqa_fixed_1000')
    demos = load_from_disk(ROOT / 'mathqa_fixed_5shot')
    assert len(test) == 1000 and len(demos) == 5
    assert not {x['Problem'].strip() for x in test} & {x['Problem'].strip() for x in demos}

    def question(row):
        return f"Question: {row['Problem'].strip()}\nOptions: {row['options'].strip()}"

    def messages(row):
        msg = [{'role': 'system', 'content': 'Solve the multiple-choice math problem. Explain your reasoning concisely, then end with exactly: Final answer: <letter> where <letter> is a, b, c, d, or e.'}]
        for d in demos:
            msg += [{'role': 'user', 'content': question(d)}, {'role': 'assistant', 'content': d['Rationale'].strip() + '\nFinal answer: ' + d['correct'].strip().lower()}]
        return msg + [{'role': 'user', 'content': question(row)}]

    def parse_answer(text):
        matches = re.findall('(?im)^\\s*(?:\\*\\*)?final answer\\s*:\\s*(?:\\*\\*)?\\s*[\\(\\[]?([a-e])\\b', text)
        if matches:
            return matches[-1].lower()
        matches = re.findall('(?i)\\b(?:the\\s+)?(?:correct\\s+)?answer\\s*(?:is|:)\\s*(?:\\*\\*)?[\\(\\[]?([a-e])\\b', text)
        if matches:
            return matches[-1].lower()
        return text.strip().lower() if re.fullmatch('[a-e]', text.strip().lower()) else None
    correct = 0
    processed = 0
    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start + 4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)
        correct += sum((parse_answer(text) == row['correct'].strip().lower() for text, row in zip(generated, rows)))
        processed += len(rows)
    return (correct, len(test))
try:
    load_quantized_model(8)
    correct, total = run_evaluation()
    print(f'MathQA | Quanto W8A16 | {correct}/{total} | accuracy: {correct / total:.2%}', flush=True)
finally:
    release_model()
'''
import os, subprocess
from pathlib import Path
from collections import deque
_root = Path('/tf/wjy')
_python = _root / '.venv-quant/bin/python'
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = '1'
_env['PATH'] = str(_python.parent) + os.pathsep + _env.get('PATH', '')
_env['PYTHONWARNINGS'] = 'ignore'
for key, folder in [('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'), ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')]:
    directory = _root / folder
    directory.mkdir(parents=True, exist_ok=True)
    _env[key] = str(directory)
_tail = deque(maxlen=25)
with subprocess.Popen([str(_python), '-u', '-c', quant_code], cwd=str(_root), env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as _p:
    try:
        for line in _p.stdout:
            _tail.append(line)
            if line.startswith(('HumanEval |', 'MathQA |', 'MMLU |')):
                print(line, end='', flush=True)
        if _p.wait() != 0:
            raise RuntimeError('评测失败，末尾诊断：' + ''.join(_tail))
    except BaseException:
        if _p.poll() is None:
            _p.terminate()
            _p.wait()
        raise

In [1]:
quant_code = r'''
import os, sysconfig
from pathlib import Path
_nvroot = Path(sysconfig.get_paths()['purelib']) / 'nvidia'
os.environ['CPATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/include')))
os.environ['LIBRARY_PATH'] = os.pathsep.join((str(p) for p in _nvroot.glob('*/lib')))
os.environ['MAX_JOBS'] = '2'
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.6;8.9'
GPU = 0
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import gc, re, ast, textwrap, warnings
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
warnings.filterwarnings('ignore', category=UserWarning)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
ROOT = Path('/tf/wjy')
MODEL_PATH = ROOT / 'models/Meta-Llama-3-8B-Instruct'
assert torch.cuda.is_available(), 'CUDA is required'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
EOS_IDS = list({tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')})

def release_model():
    global model
    if 'model' in globals():
        del model
    gc.collect()
    torch.cuda.empty_cache()

@torch.inference_mode()
def generate_batch(messages, max_new_tokens):
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False)
    assert inputs.input_ids.shape[1] + max_new_tokens <= model.config.max_position_embeddings
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    output = model.generate(**inputs, do_sample=False, temperature=None, top_p=None, max_new_tokens=max_new_tokens, eos_token_id=EOS_IDS, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(output[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def run_evaluation():
    test = load_from_disk(ROOT / 'mathqa_fixed_1000')
    demos = load_from_disk(ROOT / 'mathqa_fixed_5shot')
    assert len(test) == 1000 and len(demos) == 5
    assert not {x['Problem'].strip() for x in test} & {x['Problem'].strip() for x in demos}

    def question(row):
        return f"Question: {row['Problem'].strip()}\nOptions: {row['options'].strip()}"

    def messages(row):
        msg = [{'role': 'system', 'content': 'Solve the multiple-choice math problem. Explain your reasoning concisely, then end with exactly: Final answer: <letter> where <letter> is a, b, c, d, or e.'}]
        for d in demos:
            msg += [{'role': 'user', 'content': question(d)}, {'role': 'assistant', 'content': d['Rationale'].strip() + '\nFinal answer: ' + d['correct'].strip().lower()}]
        return msg + [{'role': 'user', 'content': question(row)}]

    def parse_answer(text):
        matches = re.findall('(?im)^\\s*(?:\\*\\*)?final answer\\s*:\\s*(?:\\*\\*)?\\s*[\\(\\[]?([a-e])\\b', text)
        if matches:
            return matches[-1].lower()
        matches = re.findall('(?i)\\b(?:the\\s+)?(?:correct\\s+)?answer\\s*(?:is|:)\\s*(?:\\*\\*)?[\\(\\[]?([a-e])\\b', text)
        if matches:
            return matches[-1].lower()
        return text.strip().lower() if re.fullmatch('[a-e]', text.strip().lower()) else None
    correct = 0
    processed = 0
    for start in range(0, len(test), 4):
        rows = [test[i] for i in range(start, min(start + 4, len(test)))]
        generated = generate_batch([messages(r) for r in rows], 1024)
        correct += sum((parse_answer(text) == row['correct'].strip().lower() for text, row in zip(generated, rows)))
        processed += len(rows)
    return (correct, len(test))

def load_awq_model():
    global model
    from datasets import load_dataset
    from awq import AutoAWQForCausalLM
    from awq.modules.linear import WQLinear_GEMM
    set_seed(42)
    raw = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train', cache_dir=str(ROOT / '.cache/huggingface/datasets'))
    calibration = []
    seen_text = set()
    for row in raw:
        text = row['text'].strip()
        ids = tokenizer.encode(text, add_special_tokens=False)
        if 32 <= len(ids) <= 512 and text not in seen_text:
            calibration.append(text)
            seen_text.add(text)
        if len(calibration) == 128:
            break
    assert len(calibration) == 128
    awq_model = AutoAWQForCausalLM.from_pretrained(str(MODEL_PATH), torch_dtype=torch.float16, trust_remote_code=False, local_files_only=True, device_map={'': 'cpu'}, attn_implementation='sdpa')
    awq_model.quantize(tokenizer, quant_config={'zero_point': True, 'q_group_size': 128, 'w_bit': 4, 'version': 'GEMM'}, calib_data=calibration, max_calib_samples=128, max_calib_seq_len=512, n_parallel_calib_samples=4, max_chunk_memory=268435456, export_compatible=False)
    model = awq_model.model.to('cuda:0').eval()
    if hasattr(awq_model, 'quantizer'):
        del awq_model.quantizer
    layers = [m for m in model.modules() if isinstance(m, WQLinear_GEMM)]
    assert len(layers) == model.config.num_hidden_layers * 7
    assert all((m.w_bit == 4 and m.qweight.dtype == torch.int32 and (m.scales.dtype == torch.float16) for m in layers))
    seen = set()

    def verify(m, args):
        assert args[0].dtype == torch.float16
        seen.add(id(m))
    hooks = [m.register_forward_pre_hook(verify) for m in layers]
    try:
        with torch.inference_mode():
            logits = model(**tokenizer('Hello', return_tensors='pt').to('cuda:0'), use_cache=False).logits
        assert len(seen) == len(layers) and torch.isfinite(logits).all()
    finally:
        for hook in hooks:
            hook.remove()
    gc.collect()
    torch.cuda.empty_cache()
try:
    load_awq_model()
    correct, total = run_evaluation()
    print(f'MathQA | AWQ W4A16 | {correct}/{total} | accuracy: {correct / total:.2%}', flush=True)
finally:
    release_model()
'''
import os, subprocess
from pathlib import Path
from collections import deque
_root = Path('/tf/wjy')
_python = _root / '.venv-quant/bin/python'
_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = '1'
_env['PATH'] = str(_python.parent) + os.pathsep + _env.get('PATH', '')
_env['PYTHONWARNINGS'] = 'ignore'
for key, folder in [('TMPDIR', '.tmp'), ('HF_HOME', '.cache/huggingface'), ('TORCH_HOME', '.cache/torch'), ('TORCH_EXTENSIONS_DIR', '.cache/torch_extensions')]:
    directory = _root / folder
    directory.mkdir(parents=True, exist_ok=True)
    _env[key] = str(directory)
_tail = deque(maxlen=25)
with subprocess.Popen([str(_python), '-u', '-c', quant_code], cwd=str(_root), env=_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as _p:
    try:
        for line in _p.stdout:
            _tail.append(line)
            if line.startswith(('HumanEval |', 'MathQA |', 'MMLU |')):
                print(line, end='', flush=True)
        if _p.wait() != 0:
            raise RuntimeError('评测失败，末尾诊断：' + ''.join(_tail))
    except BaseException:
        if _p.poll() is None:
            _p.terminate()
            _p.wait()
        raise

MathQA | AWQ W4A16 | 484/1000 | accuracy: 48.40%
